# MSR Data Layer – Interactive Demo for Lucas et al. (2025)

**Audience:** Authors of *"Effect of Salt Purity on the Corrosion of 316L SS: Long-Term Studies
in Molten FLiNaK and ThF₄–LiF"* (Lucas N. et al., *J. Nucl. Mater.* 2025, PII S0022311525007913)

This notebook walks through nine concrete scenarios that map directly to the experimental
workflow of the paper.  Each scenario demonstrates a specific capability of the deployed
MSR data layer:

| # | Scenario | API endpoint |
|---|---|---|
| 1 | Retrieve ORNL 316L / INOR-8 baselines | `POST /query` |
| 2 | Survey recent 316L / FLiNaK literature | `POST /query` |
| 3 | Ingest furnace conditions for your 18 tests | `POST /data/ingest` |
| 4 | Ingest salt-preparation & purification records | `POST /data/ingest` |
| 5 | Store ICP-OES results | `POST /data/ingest` |
| 6 | Store mass-change & SEM corrosion-depth records | `POST /data/ingest` |
| 7 | Store GIXRD phase-identification results | `POST /data/ingest` |
| 8 | Cross-experiment analysis – query the full 18-test dataset | `POST /query` |
| 9 | Future work: UF₄ / fission-product extensions | `POST /query` |

The notebook requires only the `requests` library and the deployed endpoint URL + API key.
It does **not** require any local MSR data layer installation.


## Setup

Install the only non-stdlib dependency (`requests`) if it is not already present.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'requests'])


## Configuration

Set the base URL of your MSR Data Layer deployment:

* **GitHub Codespaces (recommended):** Open the repository in a Codespace —
  the server starts automatically on port 8000 with a public URL shown in the
  *Ports* tab, e.g.
  `https://<codespace-name>-8000.app.github.dev`

* **Local:** Start the server with `make serve` (port 8000) and use
  `http://localhost:8000` as the base URL.

* **`API_KEY`** – leave as `""` unless you set `MSR_API_KEY` during startup.


In [ ]:
# ── Set your deployment URL ─────────────────────────────────────────────
# GitHub Codespaces: copy the public port-8000 URL from the Ports tab
# Local:             use http://localhost:8000
API_BASE_URL = "https://REPLACE_WITH_YOUR_CODESPACE_URL"  # e.g. https://<name>-8000.app.github.dev
API_KEY      = ""  # set only if MSR_API_KEY was configured
# ─────────────────────────────────────────────────────────────────────────


## Helper functions

A thin wrapper around the three API endpoints used in this notebook.


In [ ]:
import json
import textwrap
import requests


def _headers():
    h = {"Content-Type": "application/json"}
    if API_KEY:
        h["X-Api-Key"] = API_KEY
    return h


def health():
    """GET /health – service liveness check."""
    r = requests.get(f"{API_BASE_URL}/health", headers=_headers(), timeout=30)
    r.raise_for_status()
    return r.json()


def query(question: str, top_k: int = 5) -> dict:
    """
    POST /query – RAG knowledge-base query.

    Returns a dict with keys:
      question  – the question as submitted
      answer    – synthesised answer from the knowledge base
      top_k     – number of chunks retrieved
    """
    r = requests.post(
        f"{API_BASE_URL}/query",
        headers=_headers(),
        json={"question": question, "top_k": top_k},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()


def ingest(
    content: str,
    data_type: str = "operational_data",
    source_id: str | None = None,
) -> dict:
    """
    POST /data/ingest – add a record to the knowledge base.

    data_type must be one of:
      sensor_snapshot | event_log | maintenance_report | operational_data

    Returns a dict with keys:
      source_id    – the ID used to store this record (auto-generated if omitted)
      data_type    – as submitted
      chunks_added – number of text chunks added to the vector store
    """
    payload: dict = {"content": content, "data_type": data_type}
    if source_id:
        payload["source_id"] = source_id
    r = requests.post(
        f"{API_BASE_URL}/data/ingest",
        headers=_headers(),
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    return r.json()


def print_answer(result: dict) -> None:
    """Pretty-print a query result."""
    q = result.get("question", "")
    a = result.get("answer", "(no answer)")
    print(f"QUESTION:\n{textwrap.fill(q, 90)}")
    print()
    print("ANSWER:")
    for line in a.splitlines():
        print(textwrap.fill(line, 90) if line.strip() else "")


print("Helper functions loaded.")


---

## Step 0 – Health check

Verify the deployed service is reachable before running the scenarios.


In [ ]:
h = health()
print(f"Service : {h['service']} v{h['version']}")
print(f"Status  : {h['status']}")
print(f"KB dir  : {h['kb_dir']}")
print(f"S3 bucket: {h['s3_bucket']}")
print(f"GPU mode : {h['gpu']['local_gpu_mode']}")
print()
# Show knowledge-base source status
ds = h.get('data_source', {})
print("Knowledge-base sources:")
for k, v in ds.items():
    print(f"  {k}: {v}")


---

## Scenario 1 – Design Phase: Retrieve ORNL Baselines for 316L SS and INOR-8

**Paper connection (§1 Introduction):** Your paper benchmarks the Copenhagen Atomics 316L
data against the MSRE/MSBR heritage of Inconel and INOR-8 — roughly 1 mm corrosion per
20 000 h.  The data layer can retrieve the exact ORNL report numbers and measured values
that underpin this benchmark without manual searching across dozens of 1960s technical reports.


In [ ]:
result = query(
    "What mass-loss or corrosion-depth data exist in the ORNL reports for "
    "316 stainless steel or INOR-8 coupons in FLiNaK at 600-700 degC? "
    "Include experiment duration, temperature, and salt purity conditions."
)
print_answer(result)


In [ ]:
# Follow-up: MSRE container-material overview
result2 = query(
    "Summarise ORNL MSRE container-material corrosion data for austenitic steels. "
    "Which ORNL report numbers document the longest duration tests?"
)
print_answer(result2)


---

## Scenario 2 – Design Phase: Survey Recent 316L / FLiNaK Literature

**Paper connection (§1):** Your paper cites competing work on chromium depletion in FLiNaK
and on moisture-derived HF attack.  A single RAG query now spans six decades of literature —
ORNL reports from the 1960s *and* peer-reviewed papers from the 2010s–2020s — in one step.


In [ ]:
result = query(
    "What chromium and iron dissolution rates have been reported for 316L stainless steel "
    "in FLiNaK or FLiBe in the last 10 years? "
    "Include temperature, exposure time, and whether the salt was purified.",
    top_k=8,
)
print_answer(result)


In [ ]:
# Moisture / HF mechanism
result3 = query(
    "What role does moisture or HF play in accelerating corrosion of stainless steel "
    "in molten fluoride salts? What purification methods have been shown to reduce this effect?"
)
print_answer(result3)


---

## Scenario 3 – During Experiment: Ingest Furnace Conditions

**Paper connection (§2.1):** Your 18 immersion tests ran at 600 °C under 0.3 bar Ar
overpressure inside an argon glovebox (<10 ppm O₂ and H₂O).  The data layer stores periodic
furnace-condition records so they can be co-queried with characterisation data.

The cell below ingests a representative 4-hour reading from the purified-FLiNaK 3 000 h test.
In practice you would call this from your DAQ script every 4 hours.


In [ ]:
import json

# Sensor snapshot – call once every 4 h from the DAQ script
snapshot_text = json.dumps({
    "timestamp": "2024-06-01T04:00Z",
    "test_id": "FLiNaK-purified-3000h",
    "readings": [
        {"sensor": "furnace_temperature_c",  "value": 600.2, "unit": "degC"},
        {"sensor": "ar_overpressure_bar",    "value": 0.302, "unit": "bar"},
        {"sensor": "glovebox_o2_ppm",        "value": 7.1,   "unit": "ppm"},
        {"sensor": "glovebox_h2o_ppm",       "value": 3.4,   "unit": "ppm"},
    ],
}, indent=2)

result = ingest(
    content=snapshot_text,
    data_type="sensor_snapshot",
    source_id="FLiNaK-purified-3000h-2024-06-01T04Z",
)
print(f"Ingested sensor snapshot: source_id={result['source_id']}, chunks_added={result['chunks_added']}")


In [ ]:
# Verify: query over the ingested sensor data
result = query(
    "What was the glovebox O2 level and furnace temperature during the "
    "purified-FLiNaK 3000h test on 2024-06-01?"
)
print_answer(result)


---

## Scenario 4 – During Experiment: Ingest Salt-Preparation Records

**Paper connection (§2.1):** The key experimental variable in your paper is salt purity.
The Copenhagen Atomics purification method (high-temperature Ar treatment + HF/H₂ sparging)
is what separates the two coupon populations.  Documenting the purification batch record
for every test tube enables future statistical correlation of purity vs. corrosion depth.


In [ ]:
salt_prep_record = (
    "Salt batch CA-FLiNaK-P-007 (purified). "
    "Composition: LiF 46.5 mol%, NaF 11.5 mol%, KF 42 mol%. "
    "Purification: 24 h at 500 degC under Ar flow, followed by HF/H2 sparging. "
    "Post-purification assay: moisture below detection limit (<1 ppm), "
    "oxide impurities 6 ppm (ICP-OES). "
    "Loaded into test tube T-P-07 on 2024-05-15; "
    "four 316L coupons (IDs: P07-A, P07-B, P07-C, P07-D) suspended. "
    "Test start: 2024-05-15T10:00Z. Target exposure: 3000 h at 600 degC."
)

result = ingest(
    content=salt_prep_record,
    data_type="operational_data",
    source_id="salt-batch-CA-FLiNaK-P-007",
)
print(f"Ingested salt prep record: source_id={result['source_id']}, chunks_added={result['chunks_added']}")


In [ ]:
# Verify: trace coupons to salt batch
result = query(
    "Which coupons were loaded into test tube T-P-07, and what was the "
    "oxide impurity level of the salt batch used for that test?"
)
print_answer(result)


---

## Scenario 5 – Post-Test: Store ICP-OES Results

**Paper connection (§4.3):** Post-test salt samples were dissolved in HNO₃/HCl and
analysed by ICP-OES for Cr, Fe, and Ni.  The paper reports dissolved metal concentrations
for both purified and untreated salt conditions.

The cells below ingest representative results for one untreated-salt test tube.


In [ ]:
# ICP-OES result for untreated FLiNaK, 1000 h
icp_record = (
    "ICP-OES salt analysis — test tube T-U-03 (untreated FLiNaK, 1000 h, 600 degC). "
    "Dissolved metals: Cr 1185 mg/kg, Fe 510 mg/kg, Ni 48 mg/kg. "
    "Analysis date: 2024-07-20. Lab: Copenhagen Atomics internal. "
    "Coupons in this tube: U03-A, U03-B, U03-C, U03-D."
)
r1 = ingest(icp_record, data_type="operational_data", source_id="icp-oes/T-U-03/1000h")
print(f"{r1['source_id']}: {r1['chunks_added']} chunk(s) added")

# ICP-OES result for purified FLiNaK, 1000 h
icp_record_p = (
    "ICP-OES salt analysis — test tube T-P-07 (purified FLiNaK, 1000 h, 600 degC). "
    "Dissolved metals: Cr 108 mg/kg, Fe 19 mg/kg, Ni below detection limit. "
    "Analysis date: 2024-07-21. Lab: Copenhagen Atomics internal. "
    "Coupons in this tube: P07-A, P07-B, P07-C, P07-D."
)
r2 = ingest(icp_record_p, data_type="operational_data", source_id="icp-oes/T-P-07/1000h")
print(f"{r2['source_id']}: {r2['chunks_added']} chunk(s) added")


In [ ]:
# Verify: compare Cr levels across conditions
result = query(
    "Compare the dissolved chromium concentration in untreated versus purified FLiNaK "
    "after 1000 h exposure at 600 degC. By what factor does salt purification reduce Cr dissolution?"
)
print_answer(result)


---

## Scenario 6 – Post-Test: Store Mass-Change and SEM Corrosion-Depth Records

**Paper connection (§4.4, §4.5):** The paper reports coupon mass change (~194× greater in
untreated salt) and SEM-measured corrosion depths (untreated 68.5 µm → 112.1 µm vs.
purified 2.1 µm → 3.0 µm over 1 000–3 000 h).


In [ ]:
# Mass-change record — untreated FLiNaK, coupon U03-A, 1000 h
mass_record = (
    "Mass change — coupon U03-A (untreated FLiNaK, 1000 h, 600 degC). "
    "Pre-exposure mass: 14.823 g. Post-exposure mass: 14.695 g. "
    "Mass loss: 128 mg. Coupon area: 13.24 cm2. "
    "Specific mass loss: 9.67 mg/cm2."
)
r3 = ingest(mass_record, data_type="operational_data", source_id="mass-change/T-U-03/U03-A/1000h")
print(f"{r3['source_id']}: {r3['chunks_added']} chunk(s) added")

# SEM cross-section depth — untreated FLiNaK, coupon U03-A, 1000 h
sem_record = (
    "SEM cross-section — coupon U03-A (untreated FLiNaK, 1000 h, 600 degC). "
    "Intergranular corrosion observed. Maximum corrosion depth (ImageJ): 71 um. "
    "Mean corrosion depth: 68.5 um. "
    "Attack mode: intergranular; no uniform dissolution. "
    "Cr-depleted zone confirmed by EDS line scan."
)
r4 = ingest(sem_record, data_type="operational_data", source_id="sem/T-U-03/U03-A/1000h")
print(f"{r4['source_id']}: {r4['chunks_added']} chunk(s) added")


In [ ]:
# Verify: retrieve corrosion depth from ingested SEM record
result = query(
    "What is the mean corrosion depth measured by SEM for coupon U03-A "
    "in untreated FLiNaK after 1000 h at 600 degC? Was the attack intergranular?"
)
print_answer(result)


---

## Scenario 7 – Post-Exposure: Store GIXRD Phase-Identification Results

**Paper connection (§4.6):** GIXRD identified mechanistically important phases:
chromium carbides (Cr₇C₃, Cr₂₃C₆) on purified coupons (diffusion barrier hypothesis)
and FeCr₂O₄ spinel + γ→α-Fe transformation on untreated coupons (impurity-driven oxide attack).


In [ ]:
# GIXRD — purified FLiNaK, coupon P07-B, 3000 h
gixrd_p = (
    "GIXRD phase identification — coupon P07-B (purified FLiNaK, 3000 h, 600 degC). "
    "Phases detected: Cr7C3 (chromium carbide), Cr23C6 (chromium carbide), "
    "gamma-Fe (austenite, matrix retained). "
    "No FeCr2O4 detected. No KF or K-Cr-F compounds. "
    "Interpretation: Cr carbide surface film may act as diffusion barrier "
    "limiting further Cr dissolution into the salt."
)
r5 = ingest(gixrd_p, data_type="operational_data", source_id="gixrd/T-P-07/P07-B/3000h")
print(f"{r5['source_id']}: {r5['chunks_added']} chunk(s) added")

# GIXRD — untreated FLiNaK, coupon U03-C, 3000 h
gixrd_u = (
    "GIXRD phase identification — coupon U03-C (untreated FLiNaK, 3000 h, 600 degC). "
    "Phases detected: FeCr2O4 (spinel), KF, K-Cr-F compounds, alpha-Fe (ferrite). "
    "Original gamma-Fe austenite peak greatly reduced — consistent with "
    "Cr and Ni depletion driving gamma-to-alpha transformation. "
    "Interpretation: impurity-driven oxide dissolution removes passive Cr2O3 "
    "layer, exposing alloy to further fluoride attack."
)
r6 = ingest(gixrd_u, data_type="operational_data", source_id="gixrd/T-U-03/U03-C/3000h")
print(f"{r6['source_id']}: {r6['chunks_added']} chunk(s) added")


In [ ]:
# Verify: mechanistic comparison
result = query(
    "What GIXRD phases were found on purified FLiNaK coupons vs. untreated FLiNaK coupons "
    "after 3000 h at 600 degC? What does the presence of Cr carbides vs. FeCr2O4 spinel "
    "tell us about the corrosion mechanism in each case?"
)
print_answer(result)


---

## Scenario 8 – Cross-Experiment Analysis: Query the Full 18-Test Dataset

**Paper connection (§5 Discussion):** The paper's main finding is the ~33× difference in
corrosion depth and ~194× difference in mass loss between untreated and purified salt,
and the apparent saturation of depth in untreated salt after 2 000 h.

Once all records are ingested, the RAG pipeline synthesises cross-test comparisons in
plain language — the same analysis as §5 but queryable without spreadsheet lookups.


In [ ]:
result = query(
    "Summarise the chromium depletion depth and dissolved Cr concentration in the salt "
    "for all 316L SS coupons tested in FLiNaK, grouped by salt condition "
    "(purified vs. untreated) and exposure time. "
    "Does the untreated-salt corrosion depth appear to plateau after 2000 h?",
    top_k=10,
)
print_answer(result)


In [ ]:
result = query(
    "By what factor does salt purification reduce: (a) dissolved Cr in the salt, "
    "(b) coupon mass loss, and (c) SEM corrosion depth? "
    "Use only the data ingested for tests at 600 degC in FLiNaK."
)
print_answer(result)


---

## Scenario 9 – Future Work: Contextualising UF₄ / Fission-Product Extensions

**Paper connection (§6 Conclusion):** Section 6 lists four directions for follow-on work:
radiation effects, fission-product chemistry, UF₄ additions, and temperature/flow gradients.
The data layer provides a head-start grounded in six decades of ORNL operational data.


In [ ]:
result = query(
    "What effect did UF4 additions have on the corrosion rate of structural alloys in the MSRE? "
    "What U3+/U4+ redox ratio was maintained, and how was it controlled? "
    "Were there 316 SS or stainless steel tests in uranium-bearing fluoride salts?"
)
print_answer(result)


In [ ]:
result = query(
    "What tellurium and cesium speciation data exists for FLiNaK or FLiBe at 600-700 degC? "
    "How were fission-product impurities handled during MSRE purification cycles, "
    "and what effect did they have on structural alloy corrosion?"
)
print_answer(result)


In [ ]:
result = query(
    "What radiation effects on Hastelloy N or stainless steel have been studied in "
    "molten fluoride environments? How does neutron dose affect grain-boundary "
    "corrosion susceptibility in FLiNaK?"
)
print_answer(result)


---

## Scenario 10 – Deep Research: Comprehensive Corrosion Mechanism Analysis

**Paper connection (§1, §4–§6):** The Lucas et al. paper spans multiple
interconnected topics — salt purity effects, chromium depletion kinetics,
intergranular attack, GIXRD phase identification, and comparison with ORNL
heritage data.  A single `POST /research/deep` query now synthesises all of
these threads from the knowledge base into a long-form research report that:

* Retrieves up to 15 KB chunks per sub-query (vs. 5 for `/query`).
* Collects and returns the full set of distinct source documents cited.
* Produces a report with numbered source citations and an *Open Questions* section.

This endpoint is designed to feed a **deep research agent** that iteratively
refines its understanding of the literature before drafting a manuscript
section or a technical memo.


In [ ]:
def deep_research(question: str, top_k: int = 15) -> dict:
    """
    POST /research/deep – deep research endpoint.

    Returns a dict with keys:
      question     – the question as submitted
      report       – comprehensive long-form research report with citations
      sources      – ordered list of distinct source identifiers cited
      source_count – number of distinct sources referenced
      top_k        – number of KB chunks retrieved per sub-query
    """
    r = requests.post(
        f"{API_BASE_URL}/research/deep",
        headers=_headers(),
        json={"question": question, "top_k": top_k},
        timeout=180,
    )
    r.raise_for_status()
    return r.json()


def print_report(result: dict) -> None:
    """Pretty-print a deep research result."""
    q = result.get("question", "")
    report = result.get("report", "(no report)")
    sources = result.get("sources", [])
    print(f"RESEARCH QUESTION:\n{textwrap.fill(q, 90)}")
    print()
    print("REPORT:")
    for line in report.splitlines():
        print(textwrap.fill(line, 90) if line.strip() else "")
    print()
    print(f"SOURCES CITED ({len(sources)}):")
    for i, src in enumerate(sources, 1):
        print(f"  [{i}] {src}")


print("Deep-research helpers loaded.")


In [ ]:
# Deep research query: comprehensive corrosion mechanism analysis
# The /research/deep endpoint retrieves 15 chunks per sub-query and
# returns a long-form report with numbered citations and open questions.
result_dr = deep_research(
    "What are the dominant corrosion mechanisms for 316L stainless steel in molten "
    "FLiNaK fluoride salt at 600 degC, and how does salt purity (oxide/moisture "
    "content) control the relative contribution of intergranular chromium dissolution, "
    "oxide spinel formation, and gamma-to-alpha phase transformation? "
    "Cite ORNL heritage data and recent peer-reviewed results.",
    top_k=15,
)
print_report(result_dr)


In [ ]:
# Follow-up: deep research on salt purification methods and their quantitative effect
result_dr2 = deep_research(
    "Quantitatively compare salt purification methods (HF/H2 sparging, Ar sparging, "
    "vacuum drying) for FLiNaK and FLiBe in terms of residual oxide and moisture "
    "levels achieved, and summarise the measured reduction in corrosion rate of "
    "stainless steel or Hastelloy N coupons attributable to purification. "
    "Reference ORNL MSRE data and Copenhagen Atomics 316L SS results where available.",
    top_k=15,
)
print_report(result_dr2)


---

## Summary

This notebook demonstrated the end-to-end data-layer workflow for the Lucas et al. (2025)
experimental programme:

| Scenario | Data layer capability demonstrated |
|---|---|
| 1 – ORNL baselines | `POST /query` over ORNL archive (1960s reports) |
| 2 – Recent literature | `POST /query` over OpenAlex-ingested peer-reviewed papers |
| 3 – Furnace conditions | `POST /data/ingest` (sensor_snapshot) |
| 4 – Salt-prep records | `POST /data/ingest` (operational_data) |
| 5 – ICP-OES | `POST /data/ingest` (operational_data) |
| 6 – Mass change + SEM | `POST /data/ingest` (operational_data) |
| 7 – GIXRD phases | `POST /data/ingest` (operational_data) |
| 8 – Cross-test analysis | `POST /query` synthesising ingested records |
| 9 – UF₄ / fission-products | `POST /query` over ORNL archive for follow-on design |
| 10 – Deep research report | `POST /research/deep` – expanded RAG, numbered citations, open questions |

### Next steps

* Ingest the complete 18-test dataset (all time-points, all coupons) using the
  `ingest()` helper above.
* Use `POST /kb/update` to pull in newly published literature automatically.
* Connect an AI coding assistant (Claude Desktop, VS Code Copilot) to the `POST /mcp`
  endpoint to query the knowledge base conversationally — see
  [MSR_MCP_DEPLOYMENT_GUIDE.md](../MSR_MCP_DEPLOYMENT_GUIDE.md).

### Contact

For questions about the data layer or to request additional features, please open an issue
at https://github.com/pranavkantgaur/msr_data_layer.
